# ETL 01 — Limpeza dos dados IDEB

Leitura, filtragem e transformação dos três arquivos do IDEB municipal (INEP).

**Fontes:**
- `divulgacao_anos_iniciais_municipios_2023.xlsx`
- `divulgacao_anos_finais_municipios_2023.xlsx`
- `divulgacao_ensino_medio_municipios_2023.xlsx`

**Output:** `data/processed/ideb_series_al.parquet`

In [1]:
import sys
from pathlib import Path
import pandas as pd

# Garante que src/ é encontrado independente de onde o Jupyter foi iniciado
sys.path.insert(0, str(Path().resolve().parent))

from src.config import (
    DATA_INEP,
    DATA_PROC,
    UF_SIGLA,
    IDEB_SERIES_PARQUET,
)

INEP_NULL = ["-", "--", "*", "**", "ND"]

print("Configuração carregada.")
print(f"UF alvo: {UF_SIGLA}")
print(f"Pasta INEP: {DATA_INEP}")

Configuração carregada.
UF alvo: AL
Pasta INEP: /home/leandro/Projetos/painel_educacional_al/data/raw/inep


In [2]:
arquivo_ai = (
    DATA_INEP
    / "divulgacao_anos_iniciais_municipios_2023"
    / "divulgacao_anos_iniciais_municipios_2023.xlsx"
)

# Confirmar que header=9 é o correto
df_raw = pd.read_excel(arquivo_ai, header=None, nrows=11)
df_raw.iloc[:, :6]  # mostrar só as primeiras 6 colunas

,0,1,2,3,4,5
0,NaN,Ministério da Educação,NaN,NaN,NaN,NaN
1,NaN,Instituto Nacional de Estudos e Pesquisas Educ...,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN
3,Ensino Fundamental Regular - Anos Iniciais,NaN,NaN,NaN,NaN,NaN
4,Indicadores educacionais compostos por: Taxa d...,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN
6,Sigla da UF,Código do Município,Nome do Município,Rede,Taxa de Aprovação - 2005,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN,1º ao 5º ano,1º
9,SG_UF,CO_MUNICIPIO,NO_MUNICIPIO,REDE,VL_APROVACAO_2005_SI_4,VL_APROVACAO_2005_SI


In [7]:
ARQUIVOS = {
    "EF Anos Iniciais": DATA_INEP / "divulgacao_anos_iniciais_municipios_2023" / "divulgacao_anos_iniciais_municipios_2023.xlsx",
    "EF Anos Finais":   DATA_INEP / "divulgacao_anos_finais_municipios_2023"   / "divulgacao_anos_finais_municipios_2023.xlsx",
    "Ensino Medio":     DATA_INEP / "divulgacao_ensino_medio_municipios_2023"  / "divulgacao_ensino_medio_municipios_2023.xlsx",
}

def ler_ideb(caminho: Path, etapa: str) -> pd.DataFrame:
    df = pd.read_excel(
        caminho,
        header=9,
        dtype={"CO_MUNICIPIO": str},
        na_values=INEP_NULL,
    )

    # Filtrar AL e rede Pública
    df = df[
        (df["SG_UF"] == UF_SIGLA) &
        (df["REDE"] == "Pública")
    ].copy()

    # Apenas colunas de IDEB observado
    colunas_ideb = [c for c in df.columns if str(c).startswith("VL_OBSERVADO_")]
    df = df[["CO_MUNICIPIO", "NO_MUNICIPIO"] + colunas_ideb].copy()

    # Formato tidy
    df_tidy = df.melt(
        id_vars=["CO_MUNICIPIO", "NO_MUNICIPIO"],
        value_vars=colunas_ideb,
        var_name="ano_col",
        value_name="ideb",
    )
    df_tidy["ano"]   = df_tidy["ano_col"].str.extract(r"(\d{4})").astype(int)
    df_tidy["etapa"] = etapa
    df_tidy["ideb"]  = pd.to_numeric(df_tidy["ideb"], errors="coerce")

    return (
        df_tidy[["CO_MUNICIPIO", "NO_MUNICIPIO", "etapa", "ano", "ideb"]]
        .dropna(subset=["ideb"])
        .sort_values(["CO_MUNICIPIO", "ano"])
        .reset_index(drop=True)
    )

print("Função ler_ideb definida.")

Função ler_ideb definida.


In [8]:
# # Diagnóstico — ler sem filtrar para ver o que existe
# df_teste = pd.read_excel(
#     ARQUIVOS["EF Anos Iniciais"],
#     header=9,
#     dtype={"CO_MUNICIPIO": str},
#     na_values=INEP_NULL,
# )

# print("Valores únicos em SG_UF (primeiros 10):")
# print(df_teste["SG_UF"].unique()[:10])

# print("\nValores únicos em REDE:")
# print(df_teste["REDE"].unique())

# print("\nLinhas onde SG_UF contém 'AL':")
# print(df_teste[df_teste["SG_UF"].str.strip() == "AL"][["SG_UF", "CO_MUNICIPIO", "NO_MUNICIPIO", "REDE"]].head())

In [9]:
partes = []

for etapa, caminho in ARQUIVOS.items():
    df = ler_ideb(caminho, etapa)
    print(f"{etapa}: {df['CO_MUNICIPIO'].nunique()} municípios | anos: {sorted(df['ano'].unique())}")
    partes.append(df)

df_ideb = pd.concat(partes, ignore_index=True)
print(f"\nDataFrame final: {df_ideb.shape}")
df_ideb.head(10)

EF Anos Iniciais: 102 municípios | anos: [np.int64(2005), np.int64(2007), np.int64(2009), np.int64(2011), np.int64(2013), np.int64(2015), np.int64(2017), np.int64(2019), np.int64(2021), np.int64(2023)]
EF Anos Finais: 102 municípios | anos: [np.int64(2005), np.int64(2007), np.int64(2009), np.int64(2011), np.int64(2013), np.int64(2015), np.int64(2017), np.int64(2019), np.int64(2021), np.int64(2023)]
Ensino Medio: 98 municípios | anos: [np.int64(2017), np.int64(2019), np.int64(2021), np.int64(2023)]

DataFrame final: (2371, 5)


,CO_MUNICIPIO,NO_MUNICIPIO,etapa,ano,ideb
0,2700102,Água Branca,EF Anos Iniciais,2005,2.4
1,2700102,Água Branca,EF Anos Iniciais,2007,2.8
2,2700102,Água Branca,EF Anos Iniciais,2009,3.1
3,2700102,Água Branca,EF Anos Iniciais,2011,3.0
4,2700102,Água Branca,EF Anos Iniciais,2013,3.8
5,2700102,Água Branca,EF Anos Iniciais,2015,3.9
6,2700102,Água Branca,EF Anos Iniciais,2017,4.7
7,2700102,Água Branca,EF Anos Iniciais,2019,4.9
8,2700102,Água Branca,EF Anos Iniciais,2021,5.3
9,2700102,Água Branca,EF Anos Iniciais,2023,5.6


In [10]:
print("Etapas:", df_ideb["etapa"].unique())
print("Anos:  ", sorted(df_ideb["ano"].unique()))
print("Municípios únicos:", df_ideb["CO_MUNICIPIO"].nunique())
print("\nValores nulos por coluna:")
print(df_ideb.isnull().sum())

Etapas: <ArrowStringArray>
['EF Anos Iniciais', 'EF Anos Finais', 'Ensino Medio']
Length: 3, dtype: str
Anos:   [np.int64(2005), np.int64(2007), np.int64(2009), np.int64(2011), np.int64(2013), np.int64(2015), np.int64(2017), np.int64(2019), np.int64(2021), np.int64(2023)]
Municípios únicos: 102

Valores nulos por coluna:
CO_MUNICIPIO    0
NO_MUNICIPIO    0
etapa           0
ano             0
ideb            0
dtype: int64


In [11]:
DATA_PROC.mkdir(parents=True, exist_ok=True)
df_ideb.to_parquet(IDEB_SERIES_PARQUET, index=False)
print(f"✓ Salvo em: {IDEB_SERIES_PARQUET}")

✓ Salvo em: /home/leandro/Projetos/painel_educacional_al/data/processed/ideb_series_al.parquet


In [12]:
# Verificar que o parquet foi gravado e pode ser relido corretamente
df_check = pd.read_parquet(IDEB_SERIES_PARQUET)
print(f"Shape: {df_check.shape}")
print(f"Tipos:\n{df_check.dtypes}")
print(f"\nAmostra:")
df_check.head(5)

Shape: (2371, 5)
Tipos:
CO_MUNICIPIO        str
NO_MUNICIPIO        str
etapa               str
ano               int64
ideb            float64
dtype: object

Amostra:


,CO_MUNICIPIO,NO_MUNICIPIO,etapa,ano,ideb
0,2700102,Água Branca,EF Anos Iniciais,2005,2.4
1,2700102,Água Branca,EF Anos Iniciais,2007,2.8
2,2700102,Água Branca,EF Anos Iniciais,2009,3.1
3,2700102,Água Branca,EF Anos Iniciais,2011,3.0
4,2700102,Água Branca,EF Anos Iniciais,2013,3.8


## Decisões de limpeza registradas

- **Rede filtrada:** `Pública` (agrega Estadual + Municipal + Federal)
  — não existe linha `Total` neste dataset; `Pública` é o agregado equivalente.
- **Ensino Médio:** apenas 98 municípios e anos a partir de 2017
  — o SAEB só foi expandido para todas as escolas públicas a partir de 2017;
  municípios menores não tinham IDEB do EM antes disso. Limitação real do indicador.
- **Valores ausentes:** sentinela `"-"` do INEP convertido para `NaN` via `na_values`.
- **CO_MUNICIPIO:** lido como `str` para preservar zeros à esquerda.
- **Shape final:** 2.371 linhas × 5 colunas.

### Nota metodológica — o que representa o valor "Pública"

A coluna `REDE` nos arquivos do INEP tem três valores possíveis:
`Estadual`, `Municipal` e `Pública`.

**Estadual** e **Municipal** são calculados separadamente, usando apenas
os alunos das escolas de cada rede. **Pública** é calculado pelo INEP
a partir dos microdados de *todas* as escolas públicas do município
(estaduais + municipais + federais) tratadas como uma rede única.

Por isso o IDEB Pública **não é a soma nem a média aritmética** dos
valores Estadual e Municipal. O INEP recalcula o IDEB do zero para o
conjunto unificado, usando a fórmula `IDEB = N × P` sobre os dados
agregados de todos os alunos da rede pública. O resultado é uma média
ponderada pelo número de alunos de cada rede — a rede com mais alunos
tem mais peso no valor final.

**Consequência prática:** em municípios onde só existem escolas
municipais, o IDEB Pública é idêntico ao Municipal. Em municípios com
as duas redes, o valor Pública será dominado pela rede com mais alunos
e pode diferir dos dois valores separados.

**Por que usamos Pública:** é o indicador mais representativo da
realidade educacional do município como um todo, e é o que o próprio
INEP recomenda para comparações entre municípios. Usar apenas Estadual
ou Municipal ignoraria parte das escolas públicas do município.